# Mammography Report Classification

End-to-end pipeline: feature extraction from mammography reports, hyperparameter optimization with Optuna, multi-model comparison, and submission generation.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

try:
    from lightgbm import LGBMClassifier
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Config

In [ ]:
COMPETITION = "spr-2026-mammography-report-classification"
TARGET_COLUMN = "target"
ID_COLUMN = "ID"
N_TRIALS = 50
METRIC = "accuracy"
RANDOM_STATE = 42

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

INPUT_DIR = Path("/kaggle/input") / COMPETITION
OUTPUT_DIR = Path("/kaggle/working")

if not INPUT_DIR.exists():
    INPUT_DIR = Path("../data/raw")
    OUTPUT_DIR = Path("../data")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

## Load Data

In [ ]:
train_df = pd.read_csv(INPUT_DIR / "train.csv")
test_df = pd.read_csv(INPUT_DIR / "test.csv")

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nTarget distribution:\n{train_df[TARGET_COLUMN].value_counts().sort_index()}")
train_df.head()

## Feature Extraction

In [ ]:
ACHADOS_PATTERNS = {
    "achado_calcif_benignas": r"[Cc]alcificações benignas",
    "achado_sem_calcif_suspeitas": (
        r"[Nn]ão se observam (?:calcificações suspeitas"
        r"|microcalcificações pleomórficas|calcificações pleomórficas)"
    ),
    "achado_axilares_normais": r"regiões? axilares? não apresenta",
    "achado_mamas_lipossubst": r"[Mm]amas? parcialmente lipossubstituídas",
    "achado_mamas_densas_fibro": r"[Mm]amas? com densidades fibroglandulares",
    "achado_linfonodo_intramamario": r"[Ll]infonodo[s]? intramamário",
    "achado_assimetria": r"[Aa]ssimetria",
    "achado_nodulo": r"[Nn]ódulo",
    "achado_distorcao_arquitetural": r"[Dd]istorção arquitetural",
    "achado_calcif_vasculares": r"calcificações.*vasculares|vasculares",
    "achado_espessamento_cutaneo": r"[Ee]spessamento cutâneo",
    "achado_retracao": r"[Rr]etração",
}


def _extract_indicacao(text: str) -> str | None:
    m = re.search(
        r"Indicação clínica:\s*[\n\r]+\s*(.*?)"
        r"(?=[\n\r]+(?:Achados|Realizad|Mamas|Tecido|Calc|Impl|Alt|Desc))",
        text,
        re.DOTALL,
    )
    return m.group(1).strip() if m else None


def _classify_indicacao(text: str | None) -> str:
    if pd.isna(text) or text is None:
        return "desconhecido"
    t = text.lower().replace("\n", " ").replace("\r", " ").strip().rstrip(".")
    if "reavaliação" in t or "reavaliaçao" in t:
        return "reavaliacao"
    if "rastreamento" in t and "controle" in t:
        return "rastreamento_controle"
    if "rastreamento" in t:
        return "rastreamento"
    if "controle" in t:
        return "controle"
    if "primeiro exame" in t or "primeira mamografia" in t:
        return "primeiro_exame"
    if "rotina" in t:
        return "rotina"
    if "sintomática" in t or "nódulo palpável" in t or "queixa" in t:
        return "sintomatica"
    return "outro"


def _extract_achados(text: str) -> str | None:
    m = re.search(
        r"Achados:\s*[\n\r]+(.*?)(?=[\n\r]+Análise comparativa:|$)",
        text,
        re.DOTALL,
    )
    if m:
        return m.group(1).strip()
    m = re.search(
        r"Indicação clínica:.*?[\n\r]+.*?\.[\n\r]+(.*?)"
        r"(?=[\n\r]+Análise comparativa:|$)",
        text,
        re.DOTALL,
    )
    return m.group(1).strip() if m else None


def _extract_analise_comparativa(text: str) -> str | None:
    m = re.search(r"Análise comparativa:\s*[\n\r]+(.*?)$", text, re.DOTALL)
    return m.group(1).strip() if m else None


def extract_features(df: pd.DataFrame) -> pd.DataFrame:
    report = df["report"]
    indicacao_raw = report.apply(_extract_indicacao)
    df["indicacao_class"] = indicacao_raw.apply(_classify_indicacao)
    achados = report.apply(_extract_achados)
    for col_name, pattern in ACHADOS_PATTERNS.items():
        df[col_name] = achados.str.contains(pattern, na=False).astype(int)
    df["analise_comparativa"] = report.apply(_extract_analise_comparativa)
    return df

## Preprocessing

In [ ]:
train_df = extract_features(train_df)
test_df = extract_features(test_df)

achado_cols = list(ACHADOS_PATTERNS.keys())
feature_cols = ["indicacao_class", "analise_comparativa"] + achado_cols

X = train_df[feature_cols].copy()
y = train_df[TARGET_COLUMN].astype("uint8")
X_submission = test_df[feature_cols].copy()

numerical_columns = achado_cols
categorical_columns = ["indicacao_class"]
text_columns = ["analise_comparativa"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_columns),
        (
            "cat",
            Pipeline([("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            categorical_columns,
        ),
        (
            "txt",
            Pipeline([("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
            text_columns,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
X_sub_proc = preprocessor.transform(X_submission)

if sparse.issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
    X_test_proc = X_test_proc.toarray()
    X_sub_proc = X_sub_proc.toarray()

print(f"X_train: {X_train_proc.shape}")
print(f"X_test:  {X_test_proc.shape}")
print(f"X_sub:   {X_sub_proc.shape}")

## Model Definitions

In [ ]:
def make_model_registry(device: str) -> dict:
    use_cuda = device == "cuda"

    registry = {
        "logistic_regression": {
            "build": lambda p: LogisticRegression(solver="saga", **p),
            "params": lambda trial: {
                "C": trial.suggest_float("C", 1e-4, 10.0, log=True),
                "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
                "max_iter": trial.suggest_int("max_iter", 100, 1000, step=100),
            },
        },
        "random_forest": {
            "build": lambda p: RandomForestClassifier(n_jobs=-1, **p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "max_depth": trial.suggest_int("max_depth", 3, 30),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
            },
        },
        "gradient_boosting": {
            "build": lambda p: GradientBoostingClassifier(**p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            },
        },
        "extra_trees": {
            "build": lambda p: ExtraTreesClassifier(n_jobs=-1, **p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "max_depth": trial.suggest_int("max_depth", 3, 30),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
            },
        },
        "adaboost": {
            "build": lambda p: AdaBoostClassifier(**p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 2.0, log=True),
            },
        },
        "decision_tree": {
            "build": lambda p: DecisionTreeClassifier(**p),
            "params": lambda trial: {
                "max_depth": trial.suggest_int("max_depth", 3, 30),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
                "splitter": trial.suggest_categorical("splitter", ["best", "random"]),
            },
        },
        "svc": {
            "build": lambda p: SVC(max_iter=10000, cache_size=1000, probability=True, **p),
            "params": lambda trial: {
                "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
                "kernel": trial.suggest_categorical("kernel", ["linear", "rbf", "poly"]),
                "tol": trial.suggest_float("tol", 1e-4, 1e-2, log=True),
            },
        },
        "knn": {
            "build": lambda p: KNeighborsClassifier(n_jobs=-1, **p),
            "params": lambda trial: {
                "n_neighbors": trial.suggest_int("n_neighbors", 3, 50),
                "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
                "metric": trial.suggest_categorical("metric", ["euclidean", "manhattan", "minkowski"]),
                "p": trial.suggest_int("p", 1, 5),
            },
        },
        "naive_bayes": {
            "build": lambda p: GaussianNB(**p),
            "params": lambda trial: {
                "var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 1e-2, log=True),
            },
        },
    }

    if HAS_XGBOOST:
        xgb_base = {"verbosity": 0, "n_jobs": -1}
        if use_cuda:
            xgb_base["device"] = "cuda"
        registry["xgboost"] = {
            "build": lambda p, _b=xgb_base: XGBClassifier(**_b, **p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
            },
        }

    if HAS_LIGHTGBM:
        lgbm_base = {"verbosity": -1, "n_jobs": -1}
        if use_cuda:
            lgbm_base["device"] = "gpu"
        registry["lightgbm"] = {
            "build": lambda p, _b=lgbm_base: LGBMClassifier(**_b, **p),
            "params": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 50, 500, step=50),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "max_depth": trial.suggest_int("max_depth", 3, 15),
                "num_leaves": trial.suggest_int("num_leaves", 15, 127),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            },
        }

    if HAS_CATBOOST:
        cb_base = {"verbose": 0}
        if use_cuda:
            cb_base["task_type"] = "GPU"
        registry["catboost"] = {
            "build": lambda p, _b=cb_base: CatBoostClassifier(**_b, **p),
            "params": lambda trial: {
                "iterations": trial.suggest_int("iterations", 100, 1000, step=100),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "depth": trial.suggest_int("depth", 3, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 10.0),
                "random_strength": trial.suggest_float("random_strength", 1e-2, 10.0, log=True),
                "border_count": trial.suggest_int("border_count", 32, 255),
            },
        }

    return registry


MODEL_REGISTRY = make_model_registry(DEVICE)
print(f"Available models ({len(MODEL_REGISTRY)}): {', '.join(MODEL_REGISTRY.keys())}")

## Training & Evaluation

In [ ]:
def evaluate_multiclass(y_true, y_pred, y_proba=None):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="weighted"),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if y_proba is not None:
        if y_proba.ndim == 2:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted")
        else:
            metrics["roc_auc"] = roc_auc_score(y_true, y_proba)
        metrics["log_loss"] = log_loss(y_true, y_proba)
    return metrics


def train_model(name, spec, X_tr, y_tr, X_te, y_te, n_trials=N_TRIALS):
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    def objective(trial):
        params = spec["params"](trial)
        model = spec["build"](params)
        scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring=METRIC)
        return scores.mean()

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_params = study.best_params
    print(f"  Best CV {METRIC}: {study.best_value:.4f}")
    print(f"  Best params: {best_params}")

    model = spec["build"](best_params)
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)
    y_proba = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_te)

    metrics = evaluate_multiclass(y_te, y_pred, y_proba)
    for k, v in metrics.items():
        print(f"  Test {k}: {v:.4f}")

    return {
        "model": model,
        "best_params": best_params,
        "best_cv_score": study.best_value,
        "metrics": metrics,
        "study": study,
    }

In [ ]:
results = {}
for name, spec in tqdm(MODEL_REGISTRY.items(), desc="Models", total=len(MODEL_REGISTRY)):
    results[name] = train_model(
        name, spec, X_train_proc, y_train, X_test_proc, y_test, n_trials=N_TRIALS
    )

## Results Comparison

In [ ]:
summary_rows = []
for name, res in results.items():
    row = {"model": name, "cv_accuracy": res["best_cv_score"]}
    row.update({f"test_{k}": v for k, v in res["metrics"].items()})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("test_accuracy", ascending=False)
summary_df = summary_df.reset_index(drop=True)
summary_df.index += 1
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
summary_sorted = summary_df.sort_values("test_accuracy", ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(summary_sorted)))
ax.barh(summary_sorted["model"], summary_sorted["test_accuracy"], color=colors)
ax.set_xlabel("Test Accuracy")
ax.set_title("Model Comparison — Test Accuracy")
for i, (_, row) in enumerate(summary_sorted.iterrows()):
    ax.text(row["test_accuracy"] + 0.002, i, f"{row['test_accuracy']:.4f}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
best_name = summary_df.iloc[0]["model"]
best_model = results[best_name]["model"]
print(f"Best model: {best_name}")
print(f"Best params: {results[best_name]['best_params']}")

y_pred_best = best_model.predict(X_test_proc)
print(f"\n{classification_report(y_test, y_pred_best)}")

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, ax=ax)
ax.set_title(f"{best_name} — Confusion Matrix")
plt.tight_layout()
plt.show()

## Retrain Best Model on Full Data & Submit

In [ ]:
X_full_proc = np.vstack([X_train_proc, X_test_proc])
y_full = np.concatenate([y_train, y_test])

best_spec = MODEL_REGISTRY[best_name]
final_model = best_spec["build"](results[best_name]["best_params"])
final_model.fit(X_full_proc, y_full)
print(f"Retrained {best_name} on {X_full_proc.shape[0]} samples")

submission_preds = final_model.predict(X_sub_proc)

submission = pd.DataFrame({
    ID_COLUMN: test_df[ID_COLUMN],
    TARGET_COLUMN: submission_preds,
})

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print(f"Submission shape: {submission.shape}")
submission.head(10)